In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

In [ ]:
data = pd.read_excel("../../data1/cps12.xlsx")

In [ ]:
data.head()

Данный файл содержит чисто межобъектные данные, убедимся в этом

In [ ]:
data.year.unique()

Оценим базовую линейную модель

In [ ]:
model0 = sm.OLS.from_formula("ahe ~ age + female + bachelor", data=data).fit()
print(model0.summary())

Проведем тест Зарембки для того, чтобы понять какая объясняемая переменная нам нужна: $AHE$ или $\ln{AHE}$

In [ ]:
def zarembka(y, X, conf=0.05):
    ln_y = np.log(y)
    geom_y = np.nanmean(ln_y)

    mask = ~np.isnan(ln_y)
    n = sum(mask)

    ystar = (y / geom_y)[mask]
    ln_ystar = np.log(ystar)

    Xstar = X[mask, :]
    MX = np.eye(n) - Xstar @ np.linalg.inv(Xstar.T @ Xstar) @ Xstar.T

    SSR0 = ystar.T @ MX @ ystar
    SSR1 = ln_ystar.T @ MX @ ln_ystar

    stat = (n / 2) * np.log(max(SSR0, SSR1) / min(SSR0, SSR1))
    p_val = 1 - chi2.cdf(stat, 1)

    print(f"Zarembka stat = {stat:.3f}, p-val = {p_val:.3f}")
    if p_val > conf:
        print("Модели эквивалентны")
    else:
        print(
            "Предпочтительнее модель " + ("log(y)" if SSR1 < SSR0 else "y") + " ~ ..."
        )

In [ ]:
zarembka(model0.model.endog, model0.model.exog)

Оценим несколько спецификаций моделей для $\ln{AHE}$

In [ ]:
model1 = sm.OLS.from_formula("np.log(ahe) ~ age + female + bachelor", data=data).fit()
print(model1.summary())

Создадим примитивный датафрейм для получения подобранных значений $\widehat{\ln{AHE}}$, соответствующих мужчинам без высшего образования возраста 25, 26, 33 и 34 года.

In [ ]:
nd = pd.DataFrame({"age": [25, 26, 33, 34], "female": 0, "bachelor": 0})
ahes = np.exp(model1.predict(nd))

Убедимся, что прирост $25 \rightarrow 26$ не равен приросту $33 \rightarrow 34$

In [ ]:
ahes[1] - ahes[0], ahes[3] - ahes[2]

In [ ]:
model2 = sm.OLS.from_formula(
    "np.log(ahe) ~ np.log(age) + female + bachelor", data=data
).fit()
print(model2.summary())

In [ ]:
ahes = np.exp(model2.predict(nd))
ahes[1] - ahes[0], ahes[3] - ahes[2]

In [ ]:
model3 = sm.OLS.from_formula(
    "np.log(ahe) ~ age + I(age ** 2) + female + bachelor", data=data
).fit()
print(model3.summary())

In [ ]:
ahes = np.exp(model3.predict(nd))
ahes[1] - ahes[0], ahes[3] - ahes[2]

In [ ]:
model1.rsquared_adj, model2.rsquared_adj, model3.rsquared_adj

In [ ]:
model1.bic, model2.bic, model3.bic

In [ ]:
model1.aic, model2.aic, model3.aic

Построим графики $\widehat{\ln{AHE}}$ для возрастов от 18 до 65 лет

In [ ]:
age = np.linspace(18, 65, 1000)
nd2 = pd.DataFrame({"age": age, "female": 0, "bachelor": 0})

In [ ]:
y1 = model1.predict(nd2)
y2 = model2.predict(nd2)
y3 = model3.predict(nd2)
plt.plot(age, y1, label="log-lin")
plt.plot(age, y2, label="log-log")
plt.plot(age, y3, label="log-poly")
plt.legend()

In [ ]:
plt.plot(age, np.exp(y1), label="log-lin")
plt.plot(age, np.exp(y2), label="log-log")
plt.plot(age, np.exp(y3), label="log-poly")
plt.legend()

Добавим члены взаимодействия

In [ ]:
model4 = sm.OLS.from_formula(
    "np.log(ahe) ~ np.log(age) + female + bachelor + I(female*bachelor)", data=data
).fit()
print(model4.summary())

In [ ]:
model4 = sm.OLS.from_formula(
    "np.log(ahe) ~ np.log(age) + female + bachelor + female:bachelor", data=data
).fit()
print(model4.summary())

In [ ]:
model4 = sm.OLS.from_formula(
    "np.log(ahe) ~ np.log(age) + female*bachelor", data=data
).fit()
print(model4.summary())

In [ ]:
model1.bic, model2.bic, model3.bic, model4.bic

In [ ]:
model5 = sm.OLS.from_formula(
    "np.log(ahe) ~ (female + bachelor + np.log(age))**1", data=data
).fit()
print(model5.summary())

In [ ]:
model5 = sm.OLS.from_formula(
    "np.log(ahe) ~ (female + bachelor + np.log(age))**2", data=data
).fit()
print(model5.summary())

In [ ]:
model5 = sm.OLS.from_formula(
    "np.log(ahe) ~ (female + bachelor + np.log(age))**3", data=data
).fit()
print(model5.summary())

In [ ]:
model6 = sm.OLS.from_formula(
    "np.log(ahe) ~ np.log(age) + female*bachelor - female", data=data
).fit()
print(model6.summary())

In [ ]:
model7 = sm.OLS.from_formula(
    "np.log(ahe) ~ female + bachelor * np.log(age)", data=data
).fit()
print(model7.summary())

In [ ]:
model1.bic, model2.bic, model3.bic, model4.bic, model5.bic, model6.bic, model7.bic

# Категориальные переменные

Фиктивные переменные являются простейшим случаем категориальных переменных --- переменных, принимающих строго ограниченное число значений, зачастую не поддающихся ранжированию.
Сделаем переменную `_bachelor`, равную `bachelor`, в которой 300 произвольных значений поменяли на 2.

In [ ]:
data["_bachelor"] = data.bachelor
idx = np.random.randint(0, 7440, 300)
data.loc[idx, "_bachelor"] = 2

Если подставить ее в модель, то оценки мы получим.
Однако они предполагают, что изменение объясняемой переменной при переходе от одного значения к другому одинаково.
Это очень сильное и зачастую неправдоподобное предположение.

In [ ]:
model00 = sm.OLS.from_formula("ahe ~ age + female + _bachelor", data=data).fit()
print(model00.summary())

Для корректного учета категориальной переменной ее нужно обернуть в `C(...)`.
При этом будут автоматически созданы фиктивные переменные, соответствующие отдельным значениям переменной.

In [ ]:
model00 = sm.OLS.from_formula("ahe ~ age + female + C(_bachelor)", data=data).fit()
print(model00.summary())

Причем эти значения вообще могут быть не числовыми.

In [ ]:
data["__bachelor"] = "school"
data.loc[data._bachelor == 1, "__bachelor"] = "college"
data.loc[data._bachelor == 2, "__bachelor"] = "post-doc"

model00 = sm.OLS.from_formula("ahe ~ age + female + C(__bachelor)", data=data).fit()
print(model00.summary())